In [1]:
import pickle
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

/home/indra/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
users_path  = "../data/ml-1m/users.dat"
output_path = "../data/ml-1m/user_embeddings.pkl"
model_name  = "meta-llama/Llama-2-7b-hf"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_LENGTH = 24

In [3]:
AGE_MAP = {
    1:  "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+"
}

OCC_MAP = {
    0:  "other",
    1:  "academic/educator",
    2:  "artist",
    3:  "clerical/admin",
    4:  "college/grad student",
    5:  "customer service",
    6:  "doctor/health care",
    7:  "executive/managerial",
    8:  "farmer",
    9:  "homemaker",
    10: "K-12 student",
    11: "lawyer",
    12: "programmer",
    13: "retired",
    14: "sales/marketing",
    15: "scientist",
    16: "self-employed",
    17: "technician/engineer",
    18: "tradesman/craftsman",
    19: "unemployed",
    20: "writer"
}

In [4]:
df = pd.read_csv(
    users_path,
    sep="::",
    engine="python",
    header=None,
    encoding="latin-1",
    names=["user_id", "gender", "age", "occupation", "zipcode"]
)
print(f"{len(df)} users loaded")
df.head()

6040 users loaded


,user_id,gender,age,occupation,zipcode
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [5]:
def build_text(row):
    gender = "Male" if row["gender"] == "M" else "Female"
    age    = AGE_MAP.get(row["age"], str(row["age"]))
    occ    = OCC_MAP.get(row["occupation"], "other")
    return f"Gender: {gender}. Age: {age}. Occupation: {occ}."

df["text"] = df.apply(build_text, axis=1)
df[["user_id", "text"]].head(10)

,user_id,text
0,1,Gender: Female. Age: Under 18. Occupation: K-1...
1,2,Gender: Male. Age: 56+. Occupation: self-emplo...
2,3,Gender: Male. Age: 25-34. Occupation: scientist.
3,4,Gender: Male. Age: 45-49. Occupation: executiv...
4,5,Gender: Male. Age: 25-34. Occupation: writer.
5,6,Gender: Female. Age: 50-55. Occupation: homema...
6,7,Gender: Male. Age: 35-44. Occupation: academic...
7,8,Gender: Male. Age: 25-34. Occupation: programmer.
8,9,Gender: Male. Age: 25-34. Occupation: technici...
9,10,Gender: Female. Age: 35-44. Occupation: academ...


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

lengths = [len(tokenizer.encode(t)) for t in df["text"].tolist()]
print(f"Min: {min(lengths)}  Max: {max(lengths)}  Mean: {sum(lengths)/len(lengths):.1f}")

Min: 19  Max: 28  Mean: 23.3


In [7]:
model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16)
model = model.to(DEVICE).eval()
print(f"Loaded {model_name} on {DEVICE}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1395.64it/s]
[transformers] LlamaModel LOAD REPORT from: meta-llama/Llama-2-7b-hf
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded meta-llama/Llama-2-7b-hf on cuda


In [8]:
def get_embeddings(texts):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(embeddings.cpu().float().numpy())
        if (i // BATCH_SIZE) % 20 == 0:
            print(f"  {i + len(batch)}/{len(texts)} done")
    return np.concatenate(all_embeddings, axis=0)

print("Generating user embeddings...")
embeddings = get_embeddings(df["text"].tolist())
print(f"Embeddings shape: {embeddings.shape}")

Generating user embeddings...
  32/6040 done
  672/6040 done
  1312/6040 done
  1952/6040 done
  2592/6040 done
  3232/6040 done
  3872/6040 done
  4512/6040 done
  5152/6040 done
  5792/6040 done
Embeddings shape: (6040, 4096)


In [9]:
data = {
    "item_id":   df["user_id"].tolist(),
    "embedding": embeddings.tolist()
}

with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(data['item_id'])} user embeddings → {output_path}")
print(f"Embedding dim: {len(data['embedding'][0])}")

Saved 6040 user embeddings → ../data/ml-1m/user_embeddings.pkl
Embedding dim: 4096


## Interaction sequences + item reindexing (`ratings.dat` → `inter.json`)

The TIGER dataloader treats each item id as a **row index** into the semantic-id table
(`BatchProcessor` does `mapping[str(i)]` for `i in range(len(mapping))`), so item ids must be
**0-indexed and contiguous** (`0 .. num_items-1`).

But `index_rqvae.json` was produced keyed by the **original MovieLens movie ids** (sparse,
`1 .. 3952`). So before we can train we must:

1. Build a `movie_id → 0-indexed` map from the index's keys and **rewrite `index_rqvae.json`**
   under the new keys (values unchanged; original backed up to `index_rqvae_origids.json`).
2. Turn `ratings.dat` into per-user, timestamp-ordered sequences of those 0-indexed item ids,
   apply the core-5 filter, and save as `inter.json` (`{user_id: [item_id, ...]}`).

User ids stay as their original MovieLens ids (1..6040) — the user-embedding lookup is a dict,
not a row index, and these keys must line up with `user_embeddings.pkl` produced above.

In [ ]:
import os
import json

# --- Paths -------------------------------------------------------------------
ratings_path      = "../data/ml-1m/ratings.dat"
index_path        = "../data/ml-1m/index_rqvae.json"        # keyed by ORIGINAL movie ids
index_backup_path = "../data/ml-1m/index_rqvae_origids.json"
item_map_path     = "../data/ml-1m/item_id_map.json"        # original movie id -> 0-indexed
inter_path        = "../data/ml-1m/inter.json"

MIN_SEQ_LEN = 5   # core-5 filter (base.py asserts every user has >= 5 interactions)

# --- Load the semantic-id index ---------------------------------------------
with open(index_path) as f:
    index = json.load(f)

orig_ids = sorted(int(k) for k in index)
already_0indexed = orig_ids == list(range(len(orig_ids)))

if already_0indexed:
    # Idempotent: re-running after a previous reindex is a no-op.
    print("index_rqvae.json already 0-indexed contiguous - skipping reindex.")
    item_map = {i: i for i in orig_ids}
else:
    # Back up the original index once, before overwriting it.
    if not os.path.exists(index_backup_path):
        with open(index_backup_path, "w") as f:
            json.dump(index, f)
        print(f"Backed up original index -> {index_backup_path}")

    # original movie id -> new contiguous 0-based index (stable: sorted by orig id)
    item_map = {orig: new for new, orig in enumerate(orig_ids)}

    # Rewrite the index under the new 0-based keys (semantic codes unchanged)
    reindexed = {str(item_map[orig]): index[str(orig)] for orig in orig_ids}
    with open(index_path, "w") as f:
        json.dump(reindexed, f)
    with open(item_map_path, "w") as f:
        json.dump({str(k): v for k, v in item_map.items()}, f)
    index = reindexed
    print(f"Reindexed {len(item_map)} movies to 0..{len(item_map) - 1} -> {index_path}")
    print(f"Saved id map -> {item_map_path}")

print(f"Item vocab size: {len(index)}  (keys 0..{len(index) - 1})")

In [ ]:
from collections import defaultdict

# --- Parse ratings.dat -> per-user list of (timestamp, 0-indexed item id) ----
user_events = defaultdict(list)
n_lines = n_dropped = 0
with open(ratings_path, encoding="latin-1") as f:
    for line in f:
        n_lines += 1
        uid, mid, rating, ts = line.rstrip("\n").split("::")
        mid = int(mid)
        if mid not in item_map:      # movie has no semantic id -> skip
            n_dropped += 1
            continue
        user_events[int(uid)].append((int(ts), item_map[mid]))

# --- Chronological sequences + core-5 filter ---------------------------------
inter = {}
dropped_users = 0
for uid, events in user_events.items():
    events.sort(key=lambda e: (e[0], e[1]))   # by timestamp, tie-break on item id
    seq = [item for _, item in events]
    if len(seq) < MIN_SEQ_LEN:
        dropped_users += 1
        continue
    inter[str(uid)] = seq

with open(inter_path, "w") as f:
    json.dump(inter, f)

# --- Report ------------------------------------------------------------------
lens = [len(v) for v in inter.values()]
print(f"Ratings read:             {n_lines}")
print(f"Ratings dropped (no id):  {n_dropped}")
print(f"Users kept:               {len(inter)}  (dropped <{MIN_SEQ_LEN} items: {dropped_users})")
print(f"Seq length min/mean/max:  {min(lens)} / {sum(lens) / len(lens):.1f} / {max(lens)}")
print(f"Distinct items used:      {len({i for v in inter.values() for i in v})} / {len(index)}")
print(f"Saved -> {inter_path}")